# 🛡️ Multimodal Phishing Website Detection

**BERT (URL text) + ResNet-50 (screenshots) → phishing / legitimate classifier**

### Steps
1. ✅ Verify GPU
2. 📦 Clone repo & install dependencies
3. 📁 Mount Google Drive & point to dataset
4. ⚙️ Configure hyperparameters
5. 🔍 Inspect dataset
6. 🏋️ Train
7. 📊 Evaluate & view results

> **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import zipfile

with zipfile.ZipFile("phish360.zip", 'r') as zip_ref:
    zip_ref.extractall("data/")

---
## 1 · Verify GPU

In [1]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU detected: {gpu}  ({mem:.1f} GB VRAM)")
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → GPU.")

⚠️  No GPU found. Go to Runtime → Change runtime type → GPU.


---
## 2 · Clone repo & install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Opor123/phish-multimodal-detector.git"  # ← replace with your repo URL
REPO_DIR = "phishing_detector"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print("✅ Repo cloned.")
else:
    print("ℹ️  Repo already cloned. Pulling latest changes...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

In [ ]:
# Install all required packages (Colab already has torch/torchvision)
!pip install transformers tokenizers scikit-learn Pillow tqdm -q
print("✅ Dependencies installed.")

In [ ]:
# Make sure Python can find the project modules from any working directory
import sys
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"✅ Project root on sys.path: {PROJECT_ROOT}")

---
## 3 · Mount Google Drive & point to your dataset

Expected layout inside Drive:
```
MyDrive/
  phishing_dataset/
    trainval/
      L0001_legitimate/
        URL/url.txt
        SCREEN-SHOT/screen_shoot.png
        Label/label.txt
      P0001_phishing/
        ...
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive mounted at /content/drive")

In [ ]:
# ── ✏️  SET YOUR DATASET PATH HERE ──────────────────────────────────────────
DATASET_DIR = "/content/drive/MyDrive/phishing_dataset/trainval"
# ─────────────────────────────────────────────────────────────────────────────

if os.path.isdir(DATASET_DIR):
    sample_count = len([d for d in os.scandir(DATASET_DIR) if d.is_dir()])
    print(f"✅ Dataset found – {sample_count} sample folder(s) detected.")
else:
    print(f"❌ Path not found: {DATASET_DIR}")
    print("   Update DATASET_DIR above to match your Drive layout.")

---
## 4 · Configure hyperparameters

In [ ]:
from configs.config import config

# ── ✏️  Tweak any setting here before training ───────────────────────────────
config.DATASET_DIR        = DATASET_DIR
config.BATCH_SIZE         = 8       # lower if you hit OOM on T4 (16 GB)
config.NUM_EPOCHS         = 10
config.LEARNING_RATE      = 2e-5
config.VAL_SPLIT          = 0.2
config.MAX_URL_LENGTH     = 128
config.NUM_WORKERS        = 2       # Colab recommends 2
# ─────────────────────────────────────────────────────────────────────────────

print("Active configuration:")
print(f"  Dataset dir   : {config.DATASET_DIR}")
print(f"  Device        : {config.DEVICE}")
print(f"  Batch size    : {config.BATCH_SIZE}")
print(f"  Epochs        : {config.NUM_EPOCHS}")
print(f"  Learning rate : {config.LEARNING_RATE}")
print(f"  Val split     : {config.VAL_SPLIT}")
print(f"  Max URL len   : {config.MAX_URL_LENGTH}")

---
## 5 · Inspect dataset

In [ ]:
from transformers import AutoTokenizer
from src.dataset import PhishingDataset, build_image_transform

tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL_NAME)
transform  = build_image_transform(config.IMAGE_SIZE, config.IMAGE_MEAN, config.IMAGE_STD)

full_dataset = PhishingDataset(
    root_dir   = config.DATASET_DIR,
    tokenizer  = tokenizer,
    max_length = config.MAX_URL_LENGTH,
    transform  = transform,
)

dist = full_dataset.get_label_distribution()
total = sum(dist.values())
print(f"\nDataset summary")
print(f"  Total samples  : {total}")
print(f"  Legitimate (0) : {dist['legitimate']}  ({dist['legitimate']/total*100:.1f}%)")
print(f"  Phishing   (1) : {dist['phishing']}    ({dist['phishing']/total*100:.1f}%)")

In [ ]:
# Preview one sample
import matplotlib.pyplot as plt
import numpy as np

sample = full_dataset[0]
label_name = "phishing" if sample["label"].item() == 1 else "legitimate"

# De-normalise image for display
mean = np.array(config.IMAGE_MEAN)
std  = np.array(config.IMAGE_STD)
img  = sample["image"].permute(1, 2, 0).numpy()
img  = np.clip(img * std + mean, 0, 1)

plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.title(f"Sample: {sample['sample_id']}  |  Label: {label_name}", fontsize=12)
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"  input_ids shape    : {sample['input_ids'].shape}")
print(f"  attention_mask shape: {sample['attention_mask'].shape}")
print(f"  image shape        : {sample['image'].shape}")

---
## 6 · Train

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)

from src.dataset import build_dataloaders
from src.model   import MultimodalPhishingDetector
from src.utils   import set_seed

set_seed(config.RANDOM_SEED)

train_loader, val_loader = build_dataloaders(
    root_dir    = config.DATASET_DIR,
    val_split   = config.VAL_SPLIT,
    batch_size  = config.BATCH_SIZE,
    num_workers = config.NUM_WORKERS,
)
print(f"✅ Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ── ✏️  Freezing options (helps if GPU memory is tight) ───────────────────
FREEZE_BERT_LAYERS = 6   # 0 = fully trainable BERT, 12 = fully frozen
FREEZE_RESNET      = True  # freeze all ResNet layers except layer4
# ─────────────────────────────────────────────────────────────────────────

model = MultimodalPhishingDetector(
    text_model_name    = config.TEXT_MODEL_NAME,
    freeze_bert_layers = FREEZE_BERT_LAYERS,
    freeze_resnet      = FREEZE_RESNET,
).to(config.DEVICE)

print("\nParameter counts:")
for name, counts in model.count_parameters().items():
    print(f"  {name:<20} trainable: {counts['trainable']:>10,} / {counts['total']:>10,}")

In [ ]:
import time
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

from src.train import train_one_epoch, evaluate
from src.utils import save_checkpoint, EarlyStopping, format_metrics

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

bert_params  = list(model.text_encoder.parameters())
other_params = (
    list(model.image_encoder.parameters())
    + list(model.fusion.parameters())
    + list(model.classifier.parameters())
)
optimizer = AdamW(
    [
        {"params": bert_params,  "lr": config.LEARNING_RATE},
        {"params": other_params, "lr": config.LEARNING_RATE * 10},
    ],
    weight_decay=config.WEIGHT_DECAY,
)
scheduler     = CosineAnnealingLR(optimizer, T_max=config.NUM_EPOCHS, eta_min=1e-7)
scaler        = GradScaler(enabled=config.DEVICE.type == "cuda")
early_stopper = EarlyStopping(patience=3, min_delta=0.001)

os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
best_ckpt  = os.path.join(config.CHECKPOINT_DIR, "best_model.pt")
best_val_f1 = 0.0
history     = []

print(f"Starting training for up to {config.NUM_EPOCHS} epochs …\n")

for epoch in range(1, config.NUM_EPOCHS + 1):
    t0 = time.time()

    train_m = train_one_epoch(model, train_loader, optimizer, criterion, scaler, config.DEVICE, epoch)
    val_m   = evaluate(model, val_loader, criterion, config.DEVICE)
    scheduler.step()

    history.append({"epoch": epoch, "train": train_m, "val": val_m})

    flag = ""
    if val_m["f1"] > best_val_f1:
        best_val_f1 = val_m["f1"]
        save_checkpoint(best_ckpt, model, optimizer, scheduler, epoch, best_val_f1)
        flag = "  ← best"

    print(
        f"Epoch {epoch:02d}/{config.NUM_EPOCHS}  ({time.time()-t0:.0f}s) | "
        f"Train {format_metrics(train_m)} | "
        f"Val   {format_metrics(val_m)}{flag}"
    )

    if early_stopper(val_m["f1"]):
        print(f"\n⏹  Early stopping at epoch {epoch}.")
        break

print(f"\n✅ Training complete. Best val F1: {best_val_f1:.4f}")

---
## 7 · Evaluate & visualise results

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

epochs      = [h["epoch"]            for h in history]
train_loss  = [h["train"]["loss"]    for h in history]
val_loss    = [h["val"]["loss"]      for h in history]
train_f1    = [h["train"]["f1"]      for h in history]
val_f1      = [h["val"]["f1"]        for h in history]
train_acc   = [h["train"]["accuracy"] for h in history]
val_acc     = [h["val"]["accuracy"]   for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (train_vals, val_vals, title, ylabel) in zip(axes, [
    (train_loss, val_loss, "Loss",     "Cross-entropy loss"),
    (train_f1,   val_f1,   "F1 Score", "F1"),
    (train_acc,  val_acc,  "Accuracy", "Accuracy"),
]):
    ax.plot(epochs, train_vals, "o-", label="Train", linewidth=2)
    ax.plot(epochs, val_vals,   "s--", label="Val",   linewidth=2)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
print("Figure saved as training_curves.png")

In [ ]:
# Full classification report using the best saved checkpoint
import torch
from src.utils import load_checkpoint, full_classification_report

load_checkpoint(best_ckpt, model)
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        logits = model(
            batch["input_ids"].to(config.DEVICE),
            batch["attention_mask"].to(config.DEVICE),
            batch["image"].to(config.DEVICE),
        )
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(batch["label"].tolist())

print(full_classification_report(all_labels, all_preds))

In [ ]:
# Confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["Legitimate", "Phishing"]).plot(
    ax=ax, colorbar=False, cmap="Blues"
)
ax.set_title("Confusion Matrix (Validation Set)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Figure saved as confusion_matrix.png")

In [ ]:
# Optional: copy the best checkpoint to Drive for safekeeping
import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/phishing_detector_checkpoints"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
shutil.copy(best_ckpt, DRIVE_SAVE_DIR)
print(f"✅ Best checkpoint saved to Drive: {DRIVE_SAVE_DIR}")